## Notebook to learn to play with tif images

In [ ]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import rasterio

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

print(tf.config.list_physical_devices('GPU'))

In [ ]:
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)
# display(settings)

In [ ]:
# def permute_shuffle_batch(years, locations, batch_size, seed = 123): 
#     """this function will create two new arrays that include every permutation of 
#        dates and simulations AND where the simulation is constant for each batch. 
#        If len(dates) is not an exact multiple of batch_size, some dates will be 
#        reused to fill last batch """
    
#     np.random.seed(seed)
    
#     n = len(years)
#     r = batch_size - (n % batch_size) #remainder to fill at end
#     n2 = len(locations)
#     n_batch = (n+r)//batch_size
    
#     shuffled_dates = []
#     for i in range(n2): 
#         tmp = years
#         np.random.shuffle(tmp)
#         shuffled_dates.append(np.append(tmp, np.random.choice(tmp, r)))
        
#     shuffled_dates = np.concatenate(shuffled_dates, axis = 0).reshape(((n+r)*n2), order = 'F')
#     shuffled_simulations = np.tile(locations.repeat(batch_size), n_batch)
    
#     return shuffled_dates, shuffled_simulations

In [ ]:
imp.reload(build_data)

data_generator = build_data.data_generator(settings)

sample_years, sample_lats, sample_lons = build_data.get_shuffled_batch(settings)
print(sample_years.shape, sample_lons.shape, sample_lats.shape)

x_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))
y_tfds = tf.data.Dataset.from_tensor_slices((sample_years, sample_lats, sample_lons))

# for elem in x_tfds.take(4).as_numpy_iterator():
#     print(elem)

x_tfds = x_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=True, seed = settings["rng_seed"])
y_tfds = y_tfds.batch(settings["batch_size"]).shuffle(buffer_size = int(len(sample_years)/settings["batch_size"]), reshuffle_each_iteration=True, seed = settings["rng_seed"])

# for elem in x_tfds.take(4).as_numpy_iterator():
#     print(elem)

x_tfds = x_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_x_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

y_tfds = y_tfds.map(lambda sample_years, sample_lats, sample_lons: 
                    tf.py_function(data_generator.get_y_data, [sample_years, sample_lats, sample_lons], 
                                   Tout=tf.float64))

init_batch_x = next(x_tfds.as_numpy_iterator())

In [ ]:
imp.reload(build_model)
imp.reload(train_model)

tfds_train = tf.data.Dataset.zip((x_tfds, y_tfds))
model = build_model.build_model(settings, input_shape=np.shape(init_batch_x)[1:])

model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train.take(settings["n_batches"][0]), tfds_train.take(settings["n_batches"][1]))
